In [1]:
from src.ensemble import ScoreCombiner, StaticEnsemble

print("✓ Ensemble modules imported successfully")

✓ Ensemble modules imported successfully


In [2]:
import numpy as np

n = 1000

test_scores = {
    "if": np.random.rand(n),
    "lof": np.random.rand(n),
    "svm": np.random.rand(n),
    "ae": np.random.rand(n),
}

y_test_dummy = np.random.randint(0, 2, size=n)

ensemble = StaticEnsemble(
    weights={
        "if": 0.25,
        "lof": 0.25,
        "svm": 0.25,
        "ae": 0.25,
    },
    normalization="minmax",
)

ensemble.fit(
    validation_scores=test_scores,
    y_validation=y_test_dummy,
)

combined_scores = ensemble.anomaly_score(test_scores)
predictions = ensemble.predict(test_scores)

print("Combined scores shape:", combined_scores.shape)
print("Predictions shape:", predictions.shape)
print("Threshold:", ensemble.get_threshold())
print("Weights:", ensemble.get_weights())
print("Prediction distribution:", np.unique(
    predictions,
    return_counts=True,
))

Combined scores shape: (1000,)
Predictions shape: (1000,)
Threshold: 0.7427164391186144
Weights: {'if': 0.25, 'lof': 0.25, 'svm': 0.25, 'ae': 0.25}
Prediction distribution: (array([0, 1], dtype=int8), array([953,  47]))


In [3]:
from pathlib import Path
import joblib
import numpy as np

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "evaluation_results"

model_scores = joblib.load(
    RESULTS_DIR / "model_scores.joblib"
)

print("Available score keys:")
print(model_scores.keys())

Available score keys:
dict_keys(['if_scores', 'lof_scores', 'svm_scores', 'ae_scores'])


In [4]:
for name, values in model_scores.items():
    values = np.asarray(values)

    print("=" * 60)
    print(name)
    print("Shape :", values.shape)
    print("Dtype :", values.dtype)
    print("Min   :", values.min())
    print("Max   :", values.max())
    print("Mean  :", values.mean())
    print("Std   :", values.std())
    print("NaN   :", np.isnan(values).sum())
    print("Inf   :", np.isinf(values).sum())

if_scores
Shape : (504160,)
Dtype : float64
Min   : 0.3199917115227662
Max   : 0.7540169274246538
Mean  : 0.3806651732990593
Std   : 0.07020540947574687
NaN   : 0
Inf   : 0
lof_scores
Shape : (504160,)
Dtype : float64
Min   : -2.3174105228037583
Max   : 12616.581210068342
Mean  : 0.19251919221350167
Std   : 69.25823397425343
NaN   : 0
Inf   : 0
svm_scores
Shape : (504160,)
Dtype : float64
Min   : -75.88600571541184
Max   : 100.2054123138906
Mean  : -33.51550968443076
Std   : 20.601433085919265
NaN   : 0
Inf   : 0
ae_scores
Shape : (504160,)
Dtype : float32
Min   : 6.753342e-05
Max   : 2771.1184
Mean  : 0.12929772
Std   : 4.642928
NaN   : 0
Inf   : 0


In [5]:
from pathlib import Path
import joblib

print("Available evaluation files:")

for path in RESULTS_DIR.iterdir():
    print(path.name)

Available evaluation files:
.ipynb_checkpoints
benchmark.csv
model_predictions.joblib
evaluation_results.joblib
test_data.joblib
model_scores.joblib
metrics.csv


In [6]:
test_data = joblib.load(
    RESULTS_DIR / "test_data.joblib"
)

print("\nTest data keys:")
print(test_data.keys())

for key, value in test_data.items():
    try:
        print(key, type(value), getattr(value, "shape", None))
    except Exception:
        print(key, type(value))


Test data keys:
dict_keys(['X_test', 'y_test'])
X_test <class 'pandas.DataFrame'> (504160, 70)
y_test <class 'pandas.Series'> (504160,)


In [8]:
from src.preprocessing.loader import DataLoader

loader = DataLoader()

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Wednesday-workingHours.pcap_ISCX.csv",
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", 
]

df = loader.load_multiple(files)

print(df.shape)

2026-08-09 14:27:01 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv started.
2026-08-09 14:27:01 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Monday-WorkingHours.pcap_ISCX.csv
2026-08-09 14:27:03 | INFO     | AdaptiveRL | Loaded Monday-WorkingHours.pcap_ISCX.csv | Shape=(529918, 79)
2026-08-09 14:27:03 | INFO     | AdaptiveRL | Loading Monday-WorkingHours.pcap_ISCX.csv completed in 2.5909 seconds.
2026-08-09 14:27:03 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv started.
2026-08-09 14:27:03 | INFO     | AdaptiveRL | Reading /home/kalpe/projects/adaptive_rl_anomaly_detection/datasets/raw/Tuesday-WorkingHours.pcap_ISCX.csv
2026-08-09 14:27:05 | INFO     | AdaptiveRL | Loaded Tuesday-WorkingHours.pcap_ISCX.csv | Shape=(445909, 79)
2026-08-09 14:27:05 | INFO     | AdaptiveRL | Loading Tuesday-WorkingHours.pcap_ISCX.csv completed in 1.8458 seconds.
2026-08-09 14:27:05 | INFO     | AdaptiveRL | Lo

(2830743, 79)


In [9]:
from src.preprocessing.cleaner import DataCleaner

cleaner = DataCleaner()

df = cleaner.clean(df)

print(df.shape)

2026-08-09 14:27:24 | INFO     | AdaptiveRL | Replacing Infinite Values started.
2026-08-09 14:27:27 | INFO     | AdaptiveRL | Replacing Infinite Values completed in 2.8532 seconds.
2026-08-09 14:27:27 | INFO     | AdaptiveRL | Removing Duplicates started.
2026-08-09 14:27:37 | INFO     | AdaptiveRL | Removed 308381 duplicate rows.
2026-08-09 14:27:37 | INFO     | AdaptiveRL | Removing Duplicates completed in 10.8637 seconds.
2026-08-09 14:27:37 | INFO     | AdaptiveRL | Removing Missing Values started.
2026-08-09 14:27:38 | INFO     | AdaptiveRL | Removed 1564 rows containing missing values.
2026-08-09 14:27:38 | INFO     | AdaptiveRL | Removing Missing Values completed in 1.1882 seconds.
2026-08-09 14:27:38 | INFO     | AdaptiveRL | Removing Constant Columns started.
2026-08-09 14:27:41 | INFO     | AdaptiveRL | Removed 8 constant columns.
2026-08-09 14:27:41 | INFO     | AdaptiveRL | Removing Constant Columns completed in 2.3220 seconds.
2026-08-09 14:27:41 | INFO     | AdaptiveRL |

(2520798, 71)


In [10]:
df.columns = df.columns.str.strip()

In [11]:
from src.preprocessing.encoder import DataEncoder

encoder = DataEncoder(target_column=" Label")

df = encoder.fit_transform(df)

print(df.dtypes["Label"])
print(df["Label"].unique()[:10])

2026-08-09 14:28:00 | INFO     | AdaptiveRL | Encoding Dataset started.
2026-08-09 14:28:00 | INFO     | AdaptiveRL | Encoding Dataset completed in 0.3049 seconds.
2026-08-09 14:28:00 | INFO     | AdaptiveRL | Encoding completed.


int64
[ 0  7 11  6  5  4  3  8 12 14]


In [12]:
from src.preprocessing.scaler import DataScaler

scaler = DataScaler(
    method="standard",
    target_column="Label",
)

df = scaler.fit_transform(df)

print(df.shape)
print(df["Label"].dtype)
print(df["Label"].unique()[:10])

2026-08-09 14:28:11 | INFO     | AdaptiveRL | Scaler (standard) fitted on 70 feature columns.
2026-08-09 14:28:12 | INFO     | AdaptiveRL | Scaling Dataset started.
2026-08-09 14:28:14 | INFO     | AdaptiveRL | Scaling Dataset completed in 2.3040 seconds.
2026-08-09 14:28:14 | INFO     | AdaptiveRL | Scaling completed.


(2520798, 71)
int64
[ 0  7 11  6  5  4  3  8 12 14]


In [13]:
from pathlib import Path

PROCESSED_DIR = Path("datasets/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PATH = PROCESSED_DIR / "cicids2017_processed.parquet"

df.to_parquet(PROCESSED_PATH, index=False)

print(f"Processed dataset saved to:")
print(PROCESSED_PATH)
print(f"Shape: {df.shape}")

Processed dataset saved to:
datasets/processed/cicids2017_processed.parquet
Shape: (2520798, 71)


In [15]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Label"])
y = (df["Label"] != 0).astype(int)

X_train, X_test_recreated, y_train, y_test_recreated = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test_recreated.shape)
print("y_test:", y_test_recreated.shape)

X_train: (2016638, 70)
y_train: (2016638,)
X_test: (504160, 70)
y_test: (504160,)


In [16]:
saved_X_test = test_data["X_test"]
saved_y_test = test_data["y_test"]

print("X_test identical:",
      X_test_recreated.equals(saved_X_test))

print("y_test identical:",
      y_test_recreated.equals(saved_y_test))

X_test identical: False
y_test identical: False


In [17]:
print("=== X TEST ===")
print("Shapes:", X_test_recreated.shape, saved_X_test.shape)

print("Columns identical:",
      X_test_recreated.columns.equals(saved_X_test.columns))

print("Index identical:",
      X_test_recreated.index.equals(saved_X_test.index))

print("Values identical:",
      np.array_equal(
          X_test_recreated.to_numpy(),
          saved_X_test.to_numpy()
      ))

print("\n=== Y TEST ===")
print("Shapes:", y_test_recreated.shape, saved_y_test.shape)

print("Index identical:",
      y_test_recreated.index.equals(saved_y_test.index))

print("Values identical:",
      np.array_equal(
          y_test_recreated.to_numpy(),
          saved_y_test.to_numpy()
      ))

=== X TEST ===
Shapes: (504160, 70) (504160, 70)
Columns identical: True
Index identical: False
Values identical: True

=== Y TEST ===
Shapes: (504160,) (504160,)
Index identical: False
Values identical: True


In [18]:
X_test = X_test_recreated.reset_index(drop=True)
y_test = y_test_recreated.reset_index(drop=True)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)

print(X_test.shape)
print(y_test.shape)

(504160, 70)
(504160,)


In [19]:
from sklearn.model_selection import train_test_split

X_dev, X_val, y_dev, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.10,
    random_state=42,
    stratify=y_train,
)

print("Development:", X_dev.shape)
print("Validation :", X_val.shape)

print("\nValidation labels:")
print(y_val.value_counts().sort_index())

Development: (1814974, 70)
Validation : (201664, 70)

Validation labels:
Label
0    167605
1     34059
Name: count, dtype: int64


In [21]:
from src.models.isolation_forest import IsolationForestModel
from src.models.local_outlier_factor import LocalOutlierFactorModel
from src.models.one_class_svm import OneClassSVMModel
from src.models.autoencoder import AutoEncoderModel

MODEL_DIR = Path.cwd() / "trained_models"

if_model = IsolationForestModel.load(
    MODEL_DIR / "isolation_forest.joblib"
)

lof_model = LocalOutlierFactorModel.load(
    MODEL_DIR / "local_outlier_factor.joblib"
)

svm_model = OneClassSVMModel.load(
    MODEL_DIR / "one_class_svm.joblib"
)

ae_model = AutoEncoderModel.load(
    MODEL_DIR / "autoencoder.joblib"
)

print("✓ All four models loaded")

2026-08-09 14:37:29 | INFO     | src.models.isolation_forest | Isolation Forest loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/isolation_forest.joblib
2026-08-09 14:37:29 | INFO     | src.models.local_outlier_factor | Local Outlier Factor loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/local_outlier_factor.joblib
2026-08-09 14:37:29 | INFO     | src.models.one_class_svm | One-Class SVM loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/one_class_svm.joblib
2026-08-09 14:37:33 | INFO     | src.models.autoencoder | AutoEncoder loaded <- /home/kalpe/projects/adaptive_rl_anomaly_detection/notebooks/trained_models/autoencoder.joblib


✓ All four models loaded


In [22]:
if_val_scores = if_model.anomaly_score(X_val)
lof_val_scores = lof_model.anomaly_score(X_val)
svm_val_scores = svm_model.anomaly_score(X_val)
ae_val_scores = ae_model.anomaly_score(X_val)

validation_scores = {
    "if": np.asarray(if_val_scores),
    "lof": np.asarray(lof_val_scores),
    "svm": np.asarray(svm_val_scores),
    "ae": np.asarray(ae_val_scores),
}

for name, scores in validation_scores.items():
    print(
        f"{name}: "
        f"shape={scores.shape}, "
        f"min={scores.min():.6f}, "
        f"max={scores.max():.6f}, "
        f"mean={scores.mean():.6f}, "
        f"std={scores.std():.6f}"
    )

if: shape=(201664,), min=0.319992, max=0.744344, mean=0.380608, std=0.070081
lof: shape=(201664,), min=-2.296478, max=8985.527440, mean=0.434657, std=79.923384
svm: shape=(201664,), min=-73.646861, max=100.205412, mean=-33.470205, std=20.520261
ae: shape=(201664,), min=0.000068, max=2774.155029, mean=0.146533, std=8.841113


In [23]:
from src.ensemble import ScoreCombiner

normalizations = ["minmax", "percentile"]

normalization_results = {}

for method in normalizations:

    combiner = ScoreCombiner(
        weights={
            "if": 0.25,
            "lof": 0.25,
            "svm": 0.25,
            "ae": 0.25,
        },
        normalization=method,
    )

    combiner.fit(validation_scores)

    normalized = combiner.get_normalized_scores(
        validation_scores
    )

    normalization_results[method] = normalized

    print("=" * 60)
    print(f"Normalization: {method}")

    for name, scores in normalized.items():
        print(
            f"{name}: "
            f"min={scores.min():.4f}, "
            f"max={scores.max():.4f}, "
            f"mean={scores.mean():.4f}, "
            f"std={scores.std():.4f}"
        )

Normalization: minmax
if: min=0.0000, max=1.0000, mean=0.1428, std=0.1651
lof: min=0.0000, max=1.0000, mean=0.0003, std=0.0089
svm: min=0.0000, max=1.0000, mean=0.2311, std=0.1180
ae: min=0.0000, max=1.0000, mean=0.0001, std=0.0032
Normalization: percentile
if: min=0.0000, max=1.0000, mean=0.5000, std=0.2886
lof: min=0.0000, max=1.0000, mean=0.5000, std=0.2887
svm: min=0.0000, max=1.0000, mean=0.5000, std=0.2887
ae: min=0.0000, max=1.0000, mean=0.5000, std=0.2887


In [24]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

def evaluate_ensemble_scores(
    scores,
    y_true,
    threshold_percentile=95.0,
):
    normal_scores = scores[y_true == 0]

    threshold = np.percentile(
        normal_scores,
        threshold_percentile,
    )

    y_pred = (scores >= threshold).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true, y_pred, zero_division=0
        ),
        "recall": recall_score(
            y_true, y_pred, zero_division=0
        ),
        "f1": f1_score(
            y_true, y_pred, zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_true, scores
        ),
        "pr_auc": average_precision_score(
            y_true, scores
        ),
        "fpr": fp / (fp + tn),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


results = []

for normalization in ["minmax", "percentile"]:

    combiner = ScoreCombiner(
        weights={
            "if": 0.25,
            "lof": 0.25,
            "svm": 0.25,
            "ae": 0.25,
        },
        normalization=normalization,
    )

    combiner.fit(validation_scores)

    ensemble_scores = combiner.combine(
        validation_scores
    )

    result = evaluate_ensemble_scores(
        ensemble_scores,
        y_val.to_numpy(),
    )

    result["method"] = "Equal Weight"
    result["normalization"] = normalization

    results.append(result)


normalization_df = pd.DataFrame(results)

normalization_df[
    [
        "method",
        "normalization",
        "threshold",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "pr_auc",
        "fpr",
    ]
]

,method,normalization,threshold,precision,recall,f1,roc_auc,pr_auc,fpr
0,Equal Weight,minmax,0.215958,0.372586,0.146129,0.209925,0.747992,0.350445,0.050004
1,Equal Weight,percentile,0.806599,0.512194,0.258375,0.343482,0.780383,0.404079,0.050004


In [25]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

individual_results = []

for name, scores in validation_scores.items():

    # Use the model's existing threshold for binary prediction
    if name == "if":
        predictions = if_model.predict(X_val)
    elif name == "lof":
        predictions = lof_model.predict(X_val)
    elif name == "svm":
        predictions = svm_model.predict(X_val)
    elif name == "ae":
        predictions = ae_model.predict(X_val)

    result = {
        "model": name,
        "precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(
            y_val, scores
        ),
        "pr_auc": average_precision_score(
            y_val, scores
        ),
    }

    individual_results.append(result)

validation_model_df = pd.DataFrame(individual_results)

validation_model_df

,model,precision,recall,f1,roc_auc,pr_auc
0,if,0.337793,0.099328,0.153515,0.749830,0.339557
1,lof,0.099346,0.029449,0.045431,0.458300,0.150721
2,svm,0.258790,0.077366,0.119120,0.720339,0.303365
3,ae,0.700047,0.574503,0.631092,0.899944,0.734257


In [26]:
validation_model_df.sort_values(
    "pr_auc",
    ascending=False
)

,model,precision,recall,f1,roc_auc,pr_auc
3,ae,0.700047,0.574503,0.631092,0.899944,0.734257
0,if,0.337793,0.099328,0.153515,0.749830,0.339557
2,svm,0.258790,0.077366,0.119120,0.720339,0.303365
1,lof,0.099346,0.029449,0.045431,0.458300,0.150721


In [27]:
weights_raw = validation_model_df.set_index("model")["pr_auc"]

weights = weights_raw / weights_raw.sum()

static_weights = {
    "if": float(weights["if"]),
    "lof": float(weights["lof"]),
    "svm": float(weights["svm"]),
    "ae": float(weights["ae"]),
}

print("Static PR-AUC weights:")
for model_name, weight in static_weights.items():
    print(f"{model_name:>4}: {weight:.4f}")

print("\nWeight sum:", sum(static_weights.values()))

Static PR-AUC weights:
  if: 0.2222
 lof: 0.0986
 svm: 0.1986
  ae: 0.4806

Weight sum: 1.0


In [28]:
static_ensemble = StaticEnsemble(
    weights=static_weights,
    normalization="percentile",
)

static_ensemble.fit(
    validation_scores=validation_scores,
    y_validation=y_val.to_numpy(),
)

static_val_scores = static_ensemble.anomaly_score(
    validation_scores
)

static_val_predictions = static_ensemble.predict(
    validation_scores
)

print("Weights:")
print(static_ensemble.get_weights())

print("\nThreshold:")
print(static_ensemble.get_threshold())

print("\nPrediction distribution:")
print(
    np.unique(
        static_val_predictions,
        return_counts=True,
    )
)

Weights:
{'if': 0.22223773405664535, 'lof': 0.09864609574444297, 'svm': 0.19855027742073456, 'ae': 0.4805658927781772}

Threshold:
0.8322441061879929

Prediction distribution:
(array([0, 1], dtype=int8), array([178418,  23246]))


In [29]:
from src.evaluation.metrics import MetricsEvaluator

evaluator = MetricsEvaluator()

static_result = evaluator.evaluate(
    y_true=y_val.to_numpy(),
    y_pred=static_val_predictions,
    anomaly_scores=static_val_scores,
    model_name="Static Ensemble (PR-AUC Weighted)",
)

print(static_result.summary())

Evaluation Summary: Static Ensemble (PR-AUC Weighted)
Samples          : 201664 (Normal: 167605, Anomaly: 34059)
------------------------------------------------------------
Accuracy         : 0.8633
Precision        : 0.6395
Recall           : 0.4364
F1 Score         : 0.5188
ROC-AUC          : 0.8382
PR-AUC           : 0.5323
------------------------------------------------------------
True Positive Rate  (TPR) : 0.4364
True Negative Rate  (TNR) : 0.9500
False Positive Rate (FPR) : 0.0500
False Negative Rate (FNR) : 0.5636
------------------------------------------------------------
Confusion Matrix:
    TN: 159224   FP: 8381    
    FN: 19194    TP: 14865   


In [30]:
f1_raw = validation_model_df.set_index("model")["f1"]

f1_weights = f1_raw / f1_raw.sum()

f1_static_weights = {
    "if": float(f1_weights["if"]),
    "lof": float(f1_weights["lof"]),
    "svm": float(f1_weights["svm"]),
    "ae": float(f1_weights["ae"]),
}

print("F1-based static weights:")

for model_name, weight in f1_static_weights.items():
    print(f"{model_name:>4}: {weight:.4f}")

print("Weight sum:", sum(f1_static_weights.values()))

F1-based static weights:
  if: 0.1617
 lof: 0.0479
 svm: 0.1255
  ae: 0.6649
Weight sum: 1.0


In [31]:
f1_ensemble = StaticEnsemble(
    weights=f1_static_weights,
    normalization="percentile",
)

f1_ensemble.fit(
    validation_scores=validation_scores,
    y_validation=y_val.to_numpy(),
)

f1_val_scores = f1_ensemble.anomaly_score(
    validation_scores
)

f1_val_predictions = f1_ensemble.predict(
    validation_scores
)

f1_result = evaluator.evaluate(
    y_true=y_val.to_numpy(),
    y_pred=f1_val_predictions,
    anomaly_scores=f1_val_scores,
    model_name="Static Ensemble (F1 Weighted)",
)

print(f1_result.summary())

Evaluation Summary: Static Ensemble (F1 Weighted)
Samples          : 201664 (Normal: 167605, Anomaly: 34059)
------------------------------------------------------------
Accuracy         : 0.8787
Precision        : 0.6822
Recall           : 0.5281
F1 Score         : 0.5953
ROC-AUC          : 0.8663
PR-AUC           : 0.6279
------------------------------------------------------------
True Positive Rate  (TPR) : 0.5281
True Negative Rate  (TNR) : 0.9500
False Positive Rate (FPR) : 0.0500
False Negative Rate (FNR) : 0.4719
------------------------------------------------------------
Confusion Matrix:
    TN: 159224   FP: 8381    
    FN: 16072    TP: 17987   


In [32]:
recall_raw = validation_model_df.set_index("model")["recall"]

recall_weights = recall_raw / recall_raw.sum()

recall_static_weights = {
    "if": float(recall_weights["if"]),
    "lof": float(recall_weights["lof"]),
    "svm": float(recall_weights["svm"]),
    "ae": float(recall_weights["ae"]),
}

print("Recall-based static weights:")

for model_name, weight in recall_static_weights.items():
    print(f"{model_name:>4}: {weight:.4f}")

print("Weight sum:", sum(recall_static_weights.values()))

Recall-based static weights:
  if: 0.1272
 lof: 0.0377
 svm: 0.0991
  ae: 0.7359
Weight sum: 1.0


In [33]:
recall_ensemble = StaticEnsemble(
    weights=recall_static_weights,
    normalization="percentile",
)

recall_ensemble.fit(
    validation_scores=validation_scores,
    y_validation=y_val.to_numpy(),
)

recall_val_scores = recall_ensemble.anomaly_score(
    validation_scores
)

recall_val_predictions = recall_ensemble.predict(
    validation_scores
)

recall_result = evaluator.evaluate(
    y_true=y_val.to_numpy(),
    y_pred=recall_val_predictions,
    anomaly_scores=recall_val_scores,
    model_name="Static Ensemble (Recall Weighted)",
)

print(recall_result.summary())

Evaluation Summary: Static Ensemble (Recall Weighted)
Samples          : 201664 (Normal: 167605, Anomaly: 34059)
------------------------------------------------------------
Accuracy         : 0.8808
Precision        : 0.6870
Recall           : 0.5401
F1 Score         : 0.6047
ROC-AUC          : 0.8756
PR-AUC           : 0.6576
------------------------------------------------------------
True Positive Rate  (TPR) : 0.5401
True Negative Rate  (TNR) : 0.9500
False Positive Rate (FPR) : 0.0500
False Negative Rate (FNR) : 0.4599
------------------------------------------------------------
Confusion Matrix:
    TN: 159224   FP: 8381    
    FN: 15665    TP: 18394   
